## SRP506406

**paper:** [10.1016/j.aqrep.2025.102675](https://www.sciencedirect.com/science/article/pii/S2352513425000614) - Transcriptomic comparison between wild-caught and domesticated black tiger shrimp (Penaeus monodon) in early and late-vitellogenic broodstock females 

**date, curator:** 2026-08-14, Sara Carsanaro

**notes**
* no PMID
* all animals are adult female
* RNA selection is ribo-zero, Illumina Ribo-Zero Plus rRNA Depletion Kit and NEBNext Ultra RNA Library Prep Kit

### annotation summary

In [24]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,hepatopancreas,UBERON:0010266,arthropod hepatopancreas,perfect match
1,ovary,UBERON:0000992,ovary,perfect match
6,eyestalk,UBERON:0000020,sense organ,missing child term


In [25]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP506406"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 36/36 [00:38<00:00,  1.07s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,,,,,,hepatopancreas,adult,,,,F,,,6687,,,,,,W1H1,SAMN41241392,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1H1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
1,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,,,,,,ovary,adult,,,,F,,,6687,,,,,,W1O2,SAMN41241391,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
2,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,,,,,,ovary,adult,,,,F,,,6687,,,,,,W1O3,SAMN41241390,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
3,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,,,,,,ovary,adult,,,,F,,,6687,,,,,,W1O1,SAMN41241389,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
4,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,,,,,,hepatopancreas,adult,,,,F,,,6687,,,,,,D1H3,SAMN41241388,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
5,SRX24491497,SRP506406,Illumina NovaSeq 6000,SRS21240695,,,,,,hepatopancreas,adult,,,,F,,,6687,,,,,,D1H2,SAMN41241387,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
6,SRX24491496,SRP506406,Illumina NovaSeq 6000,SRS21240694,,,,,,eyestalk,adult,,,,F,,,6687,,,,,,D4E3,SAMN41241418,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E3,,,,stage 4,,TRANSCRIPTOMIC,cDNA
7,SRX24491495,SRP506406,Illumina NovaSeq 6000,SRS21240693,,,,,,eyestalk,adult,,,,F,,,6687,,,,,,D4E2,SAMN41241417,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E2,,,,stage 4,,TRANSCRIPTOMIC,cDNA
8,SRX24491494,SRP506406,Illumina NovaSeq 6000,SRS21240692,,,,,,eyestalk,adult,,,,F,,,6687,,,,,,D1E3,SAMN41241416,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1E3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
9,SRX24491493,SRP506406,Illumina NovaSeq 6000,SRS21240691,,,,,,eyestalk,adult,,,,F,,,6687,,,,,,D1E2,SAMN41241415,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1E2,,,,stage 1,,TRANSCRIPTOMIC,cDNA


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['eyestalk' 'hepatopancreas' 'ovary']


In [6]:

# conditional (based off infoOrgan)
library.loc[library["infoOrgan"] == "eyestalk", "anatId"] = "UBERON:0000020"
library.loc[library["infoOrgan"] == "eyestalk", "anatName"] = "sense organ"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "eyestalk", "anatAnnotationStatus"] = "missing child term"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "eyestalk", "anatBiologicalStatus"] = "not documented"

# conditional (based off infoOrgan)
library.loc[library["infoOrgan"] == "hepatopancreas", "anatId"] = "UBERON:0010266"
library.loc[library["infoOrgan"] == "hepatopancreas", "anatName"] = "arthropod hepatopancreas"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "hepatopancreas", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "hepatopancreas", "anatBiologicalStatus"] = "not documented"

# conditional (based off infoOrgan)
library.loc[library["infoOrgan"] == "ovary", "anatId"] = "UBERON:0000992"
library.loc[library["infoOrgan"] == "ovary", "anatName"] = "ovary"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "ovary", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "ovary", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,UBERON:0010266,arthropod hepatopancreas,,,,hepatopancreas,adult,perfect match,not documented,,F,,,6687,,,,,,W1H1,SAMN41241392,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1H1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
1,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,UBERON:0000992,ovary,,,,ovary,adult,perfect match,not documented,,F,,,6687,,,,,,W1O2,SAMN41241391,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
2,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,UBERON:0000992,ovary,,,,ovary,adult,perfect match,not documented,,F,,,6687,,,,,,W1O3,SAMN41241390,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
3,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,UBERON:0000992,ovary,,,,ovary,adult,perfect match,not documented,,F,,,6687,,,,,,W1O1,SAMN41241389,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
4,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,UBERON:0010266,arthropod hepatopancreas,,,,hepatopancreas,adult,perfect match,not documented,,F,,,6687,,,,,,D1H3,SAMN41241388,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
5,SRX24491497,SRP506406,Illumina NovaSeq 6000,SRS21240695,UBERON:0010266,arthropod hepatopancreas,,,,hepatopancreas,adult,perfect match,not documented,,F,,,6687,,,,,,D1H2,SAMN41241387,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
6,SRX24491496,SRP506406,Illumina NovaSeq 6000,SRS21240694,UBERON:0000020,sense organ,,,,eyestalk,adult,missing child term,not documented,,F,,,6687,,,,,,D4E3,SAMN41241418,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E3,,,,stage 4,,TRANSCRIPTOMIC,cDNA
7,SRX24491495,SRP506406,Illumina NovaSeq 6000,SRS21240693,UBERON:0000020,sense organ,,,,eyestalk,adult,missing child term,not documented,,F,,,6687,,,,,,D4E2,SAMN41241417,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E2,,,,stage 4,,TRANSCRIPTOMIC,cDNA
8,SRX24491494,SRP506406,Illumina NovaSeq 6000,SRS21240692,UBERON:0000020,sense organ,,,,eyestalk,adult,missing child term,not documented,,F,,,6687,,,,,,D1E3,SAMN41241416,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1E3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
9,SRX24491493,SRP506406,Illumina NovaSeq 6000,SRS21240691,UBERON:0000020,sense organ,,,,eyestalk,adult,missing child term,not documented,,F,,,6687,,,,,,D1E2,SAMN41241415,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1E2,,,,stage 1,,TRANSCRIPTOMIC,cDNA


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['adult']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,W1H1,SAMN41241392,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1H1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
1,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,W1O2,SAMN41241391,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
2,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,W1O3,SAMN41241390,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
3,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,W1O1,SAMN41241389,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
4,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,D1H3,SAMN41241388,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
5,SRX24491497,SRP506406,Illumina NovaSeq 6000,SRS21240695,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,D1H2,SAMN41241387,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
6,SRX24491496,SRP506406,Illumina NovaSeq 6000,SRS21240694,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,,,,,,D4E3,SAMN41241418,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E3,,,,stage 4,,TRANSCRIPTOMIC,cDNA
7,SRX24491495,SRP506406,Illumina NovaSeq 6000,SRS21240693,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,,,,,,D4E2,SAMN41241417,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E2,,,,stage 4,,TRANSCRIPTOMIC,cDNA
8,SRX24491494,SRP506406,Illumina NovaSeq 6000,SRS21240692,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,,,,,,D1E3,SAMN41241416,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1E3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
9,SRX24491493,SRP506406,Illumina NovaSeq 6000,SRS21240691,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [9]:
library.loc[:,'sex'] = 'F'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,W1H1,SAMN41241392,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1H1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
1,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,W1O2,SAMN41241391,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
2,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,W1O3,SAMN41241390,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
3,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,W1O1,SAMN41241389,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
4,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,D1H3,SAMN41241388,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
5,SRX24491497,SRP506406,Illumina NovaSeq 6000,SRS21240695,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,,,,,,D1H2,SAMN41241387,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
6,SRX24491496,SRP506406,Illumina NovaSeq 6000,SRS21240694,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,,,,,,D4E3,SAMN41241418,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E3,,,,stage 4,,TRANSCRIPTOMIC,cDNA
7,SRX24491495,SRP506406,Illumina NovaSeq 6000,SRS21240693,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,,,,,,D4E2,SAMN41241417,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E2,,,,stage 4,,TRANSCRIPTOMIC,cDNA
8,SRX24491494,SRP506406,Illumina NovaSeq 6000,SRS21240692,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,,,,,,D1E3,SAMN41241416,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1E3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
9,SRX24491493,SRP506406,Illumina NovaSeq 6000,SRS21240691,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'ribo-minus'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1H1,SAMN41241392,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1H1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
1,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O2,SAMN41241391,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
2,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O3,SAMN41241390,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
3,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O1,SAMN41241389,,,,,,,wild,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
4,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D1H3,SAMN41241388,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
5,SRX24491497,SRP506406,Illumina NovaSeq 6000,SRS21240695,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D1H2,SAMN41241387,,,,,,,domestic,,stage 1,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
6,SRX24491496,SRP506406,Illumina NovaSeq 6000,SRS21240694,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D4E3,SAMN41241418,,,,,,,domestic,,stage 4,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E3,,,,stage 4,,TRANSCRIPTOMIC,cDNA
7,SRX24491495,SRP506406,Illumina NovaSeq 6000,SRS21240693,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,"Illumina Ribo-Zer

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [12]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''
library.loc[library["physiologicalStatus"] == "stage 1", "physiologicalStatus"] = "previtellogenic stage"
library.loc[library["physiologicalStatus"] == "stage 4", "physiologicalStatus"] = "late cortical rod stage"

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1H1,SAMN41241392,,,,,,,wild,,previtellogenic stage,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1H1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
1,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O2,SAMN41241391,,,,,,,wild,,previtellogenic stage,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
2,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O3,SAMN41241390,,,,,,,wild,,previtellogenic stage,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
3,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O1,SAMN41241389,,,,,,,wild,,previtellogenic stage,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,W1O1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
4,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D1H3,SAMN41241388,,,,,,,domestic,,previtellogenic stage,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
5,SRX24491497,SRP506406,Illumina NovaSeq 6000,SRS21240695,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D1H2,SAMN41241387,,,,,,,domestic,,previtellogenic stage,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D1H2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
6,SRX24491496,SRP506406,Illumina NovaSeq 6000,SRS21240694,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D4E3,SAMN41241418,,,,,,,domestic,,late cortical rod stage,,14/08/2026,RNA extraction using TRIzol for library preparation then sequencing,,D4E3,,,,stage 4,,TRANSCRIPTOMIC,cDNA
7,SRX24491495,SRP506406,Illumina NovaSeq 6000,SRS21240693,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adu

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [13]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-08-14'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1H1,SAMN41241392,,,,,,,wild,,previtellogenic stage,SAC,2026-08-14,RNA extraction using TRIzol for library preparation then sequencing,,W1H1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
1,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O2,SAMN41241391,,,,,,,wild,,previtellogenic stage,SAC,2026-08-14,RNA extraction using TRIzol for library preparation then sequencing,,W1O2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
2,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O3,SAMN41241390,,,,,,,wild,,previtellogenic stage,SAC,2026-08-14,RNA extraction using TRIzol for library preparation then sequencing,,W1O3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
3,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O1,SAMN41241389,,,,,,,wild,,previtellogenic stage,SAC,2026-08-14,RNA extraction using TRIzol for library preparation then sequencing,,W1O1,,,,stage 1,,TRANSCRIPTOMIC,cDNA
4,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D1H3,SAMN41241388,,,,,,,domestic,,previtellogenic stage,SAC,2026-08-14,RNA extraction using TRIzol for library preparation then sequencing,,D1H3,,,,stage 1,,TRANSCRIPTOMIC,cDNA
5,SRX24491497,SRP506406,Illumina NovaSeq 6000,SRS21240695,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D1H2,SAMN41241387,,,,,,,domestic,,previtellogenic stage,SAC,2026-08-14,RNA extraction using TRIzol for library preparation then sequencing,,D1H2,,,,stage 1,,TRANSCRIPTOMIC,cDNA
6,SRX24491496,SRP506406,Illumina NovaSeq 6000,SRS21240694,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D4E3,SAMN41241418,,,,,,,domestic,,late cortical rod stage,SAC,2026-08-14,RNA extraction using TRIzol for library preparation then sequencing,,D4E3,,,,stage 4,,TRANSCRIPTOMIC,cDNA
7,SRX24491495,SRP506406,Illumina NovaSeq 6000,SRS21240693,UBERON:0000020,sense organ,UBERON:0000

#### comments

In [14]:
library.loc[:,'comment'] = 'no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614'

#### save complete file with correct columns

In [15]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1H1,SAMN41241392,,,,,,,"no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614",,previtellogenic stage,SAC,2026-08-14
1,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O2,SAMN41241391,,,,,,,"no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614",,previtellogenic stage,SAC,2026-08-14
2,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O3,SAMN41241390,,,,,,,"no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614",,previtellogenic stage,SAC,2026-08-14
3,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,W1O1,SAMN41241389,,,,,,,"no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614",,previtellogenic stage,SAC,2026-08-14
4,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D1H3,SAMN41241388,,,,,,,"no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614",,previtellogenic stage,SAC,2026-08-14
5,SRX24491497,SRP506406,Illumina NovaSeq 6000,SRS21240695,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D1H2,SAMN41241387,,,,,,,"no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614",,previtellogenic stage,SAC,2026-08-14
6,SRX24491496,SRP506406,Illumina NovaSeq 6000,SRS21240694,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D4E3,SAMN41241418,,,,,,,"no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614",,late cortical rod stage,SAC,2026-08-14
7,SRX24491495,SRP506406,Illumina NovaSeq 6000,SRS21240693,UBERON:0000020,sense organ,UBERON:0000113,post-juvenile adult stage,,eyestalk,adult,missing child term,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,ribo-minus,,,D4E2,SAMN41241417,,,,,,,"no PMID, https://www.sciencedirect.com/science/article/pii/S2352513425000614",,late cortical rod stage,SAC,2026-08-14
8,SRX24491494,SRP506406,Il

### experiment annotations

In [16]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP506406,Reproductive traits between wild and captive populations of female black tiger shrimp (Penaeus monodon),"The current project analyzed the transcriptomes of domesticated and natural P. monodon breeders from different stages using Illumina sequencing technology. Based on the transcriptome datasets, analysis of gene expression by bioinformatic tools was performed to identify the genes involved in development and maturation of the ovary of the black tiger shrimps. The objective of our project is to understand the biological pathways and to discover candidate genes involved in reproduction in P. monodon. This resource will be exploited in future breeding programs to improve reproductive traits in P. monodon.",SRA,,,,,,,PRJNA1108448,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [17]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

36

In [18]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP506406,Reproductive traits between wild and captive populations of female black tiger shrimp (Penaeus monodon),"The current project analyzed the transcriptomes of domesticated and natural P. monodon breeders from different stages using Illumina sequencing technology. Based on the transcriptome datasets, analysis of gene expression by bioinformatic tools was performed to identify the genes involved in development and maturation of the ovary of the black tiger shrimps. The objective of our project is to understand the biological pathways and to discover candidate genes involved in reproduction in P. monodon. This resource will be exploited in future breeding programs to improve reproductive traits in P. monodon.",SRA,total,Bgee 1K,36,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,,PRJNA1108448,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [19]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://www.sciencedirect.com/science/article/pii/S2352513425000614'
experiment.loc[:,'DOI'] = '10.1016/j.aqrep.2025.102675'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP506406,Reproductive traits between wild and captive populations of female black tiger shrimp (Penaeus monodon),"The current project analyzed the transcriptomes of domesticated and natural P. monodon breeders from different stages using Illumina sequencing technology. Based on the transcriptome datasets, analysis of gene expression by bioinformatic tools was performed to identify the genes involved in development and maturation of the ovary of the black tiger shrimps. The objective of our project is to understand the biological pathways and to discover candidate genes involved in reproduction in P. monodon. This resource will be exploited in future breeding programs to improve reproductive traits in P. monodon.",SRA,total,Bgee 1K,36,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,,PRJNA1108448,,https://www.sciencedirect.com/science/article/pii/S2352513425000614,10.1016/j.aqrep.2025.102675,,


#### comments

In [20]:
experiment.loc[:,'comment'] = 'no PMID'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP506406,Reproductive traits between wild and captive populations of female black tiger shrimp (Penaeus monodon),"The current project analyzed the transcriptomes of domesticated and natural P. monodon breeders from different stages using Illumina sequencing technology. Based on the transcriptome datasets, analysis of gene expression by bioinformatic tools was performed to identify the genes involved in development and maturation of the ovary of the black tiger shrimps. The objective of our project is to understand the biological pathways and to discover candidate genes involved in reproduction in P. monodon. This resource will be exploited in future breeding programs to improve reproductive traits in P. monodon.",SRA,total,Bgee 1K,36,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEBNext Ultra RNA Library Prep Kit",full_length,,PRJNA1108448,,https://www.sciencedirect.com/science/article/pii/S2352513425000614,10.1016/j.aqrep.2025.102675,,no PMID


#### save complete file

In [21]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [22]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [23]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [26]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [27]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
69257,SRX25395862,SRP521117,Illumina NovaSeq 6000,SRS22059082,UBERON:0010266,arthropod hepatopancreas,UBERON:0000092,post-embryonic stage,,hepatopancrea,NA,perfect match,not documented,perfect match,NA,,,6687,Illumina Stranded mRNA Prep,full_length,polyA,,,L1EFG121859--SC_Hep_1,SAMN42622784,4.56 ± 0.57,g,,,,,"PMID: 39234176, stage not clear but confirmed ...",,,SAC,2026-08-14
69258,SRX25395870,SRP521117,Illumina NovaSeq 6000,SRS22059090,CL:0000387,hemocyte (sensu Arthropoda),UBERON:0000092,post-embryonic stage,,hemocyte,NA,perfect match,not documented,perfect match,NA,,,6687,Illumina Stranded mRNA Prep,full_length,polyA,,,L1EFH270424--SC_Hea_1,SAMN42622840,4.56 ± 0.57,g,,,,,"PMID: 39234176, stage not clear but confirmed ...",,,SAC,2026-08-14
69259,SRX24491502,SRP506406,Illumina NovaSeq 6000,SRS21240700,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEB...",full_length,ribo-minus,,,W1H1,SAMN41241392,,,,,,,"no PMID, https://www.sciencedirect.com/science...",,previtellogenic stage,SAC,2026-08-14
69260,SRX24491501,SRP506406,Illumina NovaSeq 6000,SRS21240699,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEB...",full_length,ribo-minus,,,W1O2,SAMN41241391,,,,,,,"no PMID, https://www.sciencedirect.com/science...",,previtellogenic stage,SAC,2026-08-14
69261,SRX24491500,SRP506406,Illumina NovaSeq 6000,SRS21240698,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEB...",full_length,ribo-minus,,,W1O3,SAMN41241390,,,,,,,"no PMID, https://www.sciencedirect.com/science...",,previtellogenic stage,SAC,2026-08-14
69262,SRX24491499,SRP506406,Illumina NovaSeq 6000,SRS21240697,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,ovary,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEB...",full_length,ribo-minus,,,W1O1,SAMN41241389,,,,,,,"no PMID, https://www.sciencedirect.com/science...",,previtellogenic stage,SAC,2026-08-14
69263,SRX24491498,SRP506406,Illumina NovaSeq 6000,SRS21240696,UBERON:0010266,arthropod hepatopancreas,UBERON:0000113,post-juvenile adult stage,,hepatopancreas,adult,perfect match,not documented,perfect match,F,,,6687,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEB...",full_length,ribo-minus,,,D1H3,SAMN41241388,,,,,,,"no PMID, https://www.sciencedirect.com/science...",,previtellogenic stage,SAC,2026-08-14


In [28]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1325,SRP457425,Study on Low Temperature of Litopenaeus vannamei,Abstract:Temperature is a limiting factor for ...,SRA,partial,Bgee 1K,3,,,,PRJNA1010528,38295537,https://www.sciencedirect.com/science/article/...,10.1016/j.cbd.2024.101196,,"rejected cold stressed samples, kept only cont..."
1326,SRP521117,Transcriptomic analysis of Penaeus monodon in ...,To investigate the different mechanisms of Pen...,SRA,partial,Bgee 1K,9,Illumina Stranded mRNA Prep,full_length,,PRJNA1137583,39234176,https://pmc.ncbi.nlm.nih.gov/articles/PMC11371...,10.3389/fvets.2024.1464291,,"removed low salinity samples, kept only controls"
1327,SRP506406,Reproductive traits between wild and captive p...,The current project analyzed the transcriptome...,SRA,total,Bgee 1K,36,"Illumina Ribo-Zero Plus rRNA Depletion Kit,NEB...",full_length,,PRJNA1108448,,https://www.sciencedirect.com/science/article/...,10.1016/j.aqrep.2025.102675,,no PMID


### add annotations to git

In [29]:
! git pull

Already up to date.


In [ ]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git add $git_experiment_path $git_library_path

In [ ]:
! git commit -m $commit_message_exp

In [ ]:
! git push

### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push